# M1 check — reference adapters

Equipment check for `tools/annotator/refs/`. Three questions, answered on the real 59 locations:

1. **Did every adapter run everywhere?** Coverage and failures.
2. **Is the georeference right?** The `osm` adapter re-draws the same OSM ways the dataset build drew into `label.png`. Rasterised back, it must match `label.png` (shifted by the recorded `label_offset` where the adapter applied it). This also settles the **sign of the offset**.
3. **Did porting change behaviour?** The model adapters must reproduce the PoC numbers on the PoC's 9 locations: woody/water are raw masks (should match to ~1e-3), buildings are simplified polygons now (looser).

Run from `src/`: `papermill tools/annotator/checks/m1_refs.ipynb tools/annotator/checks/runs/m1_refs.ipynb`

In [ ]:
# parameters
n_poc = 9            # the PoC locations = first 9 of index.json['built']
fig_locations = 3

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
assert Path('datasets').is_dir(), 'run from src/'
sys.path.insert(0, '.')
from tools.annotator import config
from tools.annotator.geo import Grid
from tools.annotator.refs import REGISTRY, base, vectorize
Image.MAX_IMAGE_PIXELS = None
FIGS = Path('tools/annotator/checks/figures'); FIGS.mkdir(exist_ok=True)
locs = config.locations()
OSM_CLASSES = {'background': 0, 'building': 1, 'road': 2, 'water': 3, 'wood': 4}   # label.png ids
print(len(locs), 'locations;', 'sources:', list(REGISTRY))

## 1. Coverage

In [ ]:
rows = []
for src in REGISTRY:
    metas = [base.read_meta(src, s) for s in locs]
    ok = [m for m in metas if m]
    secs = [m['seconds'] for m in ok]
    classes = sorted({c for m in ok for c in m['classes']})
    rows.append((src, len(ok), len(locs), f"{np.median(secs):.1f}s", ', '.join(classes)))
print(f"{'source':15s} {'ran':>7s}  median  classes")
for r in rows: print(f"{r[0]:15s} {r[1]:>3d}/{r[2]:<3d}  {r[3]:>6s}  {r[4]}")
assert all(r[1] == r[2] for r in rows), 'some adapter did not cover every location'

## 2. Georeference: `osm` adapter vs the dataset's own `label.png`

`label.png` was drawn by PIL from the same cached Overpass response, **without** the offset. The adapter applies `label_offset` only when `peak_z >= 4` and not at the search limit, so the comparison target is `label.png` rolled by `(dx, dy)` for those locations and unrolled otherwise. Buildings are the cleanest class for this (closed polygons, no width guesses).

In [ ]:
def osm_building_iou(slug):
    meta = config.meta(slug); grid = Grid.from_meta(meta); shape = grid.shape
    lab = np.array(Image.open(config.DATASET / slug / 'label.png')) == OSM_CLASSES['building']
    rm = base.read_meta('osm', slug)
    dx, dy = rm['info']['offset_applied_px']
    target = np.roll(np.roll(lab, int(dy), 0), int(dx), 1) if (dx or dy) else lab
    unshifted = lab
    feats = [f for f in base.read_vectors('osm', slug) if f['properties']['class'] == 'building']
    mine = vectorize.rasterize(feats, grid, shape, meta['mpp'])
    iou = lambda a, b: (a & b).sum() / max(1, (a | b).sum())
    return iou(mine, target), iou(mine, unshifted), rm['info']['offset_trusted'], (dx, dy)

res = {s: osm_building_iou(s) for s in locs}
ious = np.array([r[0] for r in res.values()])
trusted = [s for s, r in res.items() if r[2]]
print(f"building IoU vs label.png (offset-matched): median {np.median(ious):.3f}, min {ious.min():.3f}, "
      f"{(ious >= 0.9).sum()}/{len(ious)} >= 0.90")
print(f"offset applied at {len(trusted)}/{len(locs)} locations")
sign_ok = sum(res[s][0] > res[s][1] for s in trusted if res[s][3] != (0, 0))
print(f"sign check: shifted target beats unshifted at {sign_ok}/{sum(res[s][3] != (0, 0) for s in trusted)} shifted locations")
for s, r in sorted(res.items(), key=lambda kv: kv[1][0])[:5]:
    print(f"  lowest: {s:32s} IoU {r[0]:.3f} (vs unshifted {r[1]:.3f}) offset {r[3]}")

The residual below 1.0 is expected and small: PIL (`label.png`) and cv2 (adapter) rasterise polygon edges differently by up to one pixel, which matters for 5 m houses. What must hold: median well above 0.9, and the **shifted** target winning wherever an offset was applied -- otherwise the sign is wrong and every OSM layer would be pushed *away* from the imagery.

## 3. Port fidelity: model adapters vs the PoC results

In [ ]:
built = json.loads((config.DATASET / 'index.json').read_text())['built'][:n_poc]
def osm_mask(slug, cls):
    # exactly the PoC helper: label.png class, rolled by label_offset (always)
    meta = config.meta(slug)
    m = np.array(Image.open(config.DATASET / slug / 'label.png')) == OSM_CLASSES[cls]
    off = meta.get('label_offset') or {}
    return np.roll(np.roll(m, int(off.get('dy_px', 0)), 0), int(off.get('dx_px', 0)), 1)
iou = lambda a, b: float((a & b).sum() / max(1, (a | b).sum()))

poc = {
  'flair': {r['location']: r for r in json.loads(Path('pocs/flair-landcover-transfer/results.json').read_text()) if r['mode'] == 'native'},
  'landcoverai': {r['location']: r for r in json.loads(Path('pocs/landcoverai-finetune/results-transfer.json').read_text())},
  'ms-footprints': {r['location']: r for r in json.loads(Path('pocs/ms-building-footprints/results.json').read_text())},
}
table = []
for slug in built:
    meta = config.meta(slug); grid = Grid.from_meta(meta)
    for src in ('flair', 'landcoverai'):
        p = poc[src][slug]
        for cls, ocls in (('woody', 'wood'), ('water', 'water')):
            m = base.read_mask(src, slug, cls); m = np.zeros(grid.shape, bool) if m is None else m
            table.append((src, cls, slug, iou(m, osm_mask(slug, ocls)), p[cls]['iou']))
        b = vectorize.rasterize([f for f in base.read_vectors(src, slug) if f['properties']['class'] == 'building'], grid, grid.shape, meta['mpp'])
        table.append((src, 'building', slug, iou(b, osm_mask(slug, 'building')), p['building']['iou']))
    b = vectorize.rasterize(base.read_vectors('ms-footprints', slug), grid, grid.shape, meta['mpp'])
    table.append(('ms-footprints', 'building', slug, iou(b, osm_mask(slug, 'building')), poc['ms-footprints'][slug]['iou']))

import collections
agg = collections.defaultdict(list)
for src, cls, slug, now, then in table: agg[(src, cls)].append((now, then))
print(f"{'source':14s} {'class':9s} {'median now':>10s} {'median PoC':>10s} {'max |diff|':>10s}")
for (src, cls), v in agg.items():
    now, then = np.array(v).T
    print(f"{src:14s} {cls:9s} {np.median(now):10.3f} {np.median(then):10.3f} {np.abs(now - then).max():10.3f}")

**Reading it.** `woody`/`water` from FLAIR and LandCover.ai are the raw argmax masks, so they should equal the PoC to rounding -- a large diff there means the port changed the model input (normalisation, tiling, resize). `building` now passes through `mask -> polygons (DP 0.5 m, min 8 m^2) -> raster`, and MS is burnt by cv2 instead of PIL and clipped to the image; small shifts are the cost of having editable polygons.

## 4. What the annotator will show: every source on 3 locations

In [ ]:
pal = {'building': (255, 59, 92), 'track': (255, 176, 32), 'woody': (46, 204, 64), 'field': (212, 225, 87), 'water': (30, 144, 255)}
def panel(ax, slug, src):
    meta = config.meta(slug); grid = Grid.from_meta(meta)
    img = np.array(Image.open(config.DATASET / slug / 'current.jpg').convert('RGB')).astype(float)
    over = img.copy()
    for cls, col in pal.items():
        m = base.read_mask(src, slug, cls)
        fs = [f for f in base.read_vectors(src, slug) if f['properties']['class'] == cls]
        v = vectorize.rasterize(fs, grid, grid.shape, meta['mpp']) if fs else None
        mm = m if v is None else (v if m is None else (m | v))
        if mm is not None: over[mm] = over[mm] * 0.35 + np.array(col) * 0.65
    ax.imshow(over[::3, ::3].astype(np.uint8)); ax.set_title(f'{src}', fontsize=9); ax.axis('off')
pick = [s for s in built if base.read_meta('ms-footprints', s)['stats']][:fig_locations]
srcs = list(REGISTRY)
fig, axes = plt.subplots(len(pick), len(srcs) + 1, figsize=(3.2 * (len(srcs) + 1), 2.6 * len(pick)))
for i, slug in enumerate(pick):
    axes[i, 0].imshow(np.array(Image.open(config.DATASET / slug / 'current.jpg'))[::3, ::3]); axes[i, 0].set_title(slug, fontsize=8); axes[i, 0].axis('off')
    for j, src in enumerate(srcs): panel(axes[i, j + 1], slug, src)
plt.tight_layout(); plt.savefig(FIGS / 'm1-sources.jpg', dpi=110); plt.show()